# Creating GRN using SCENIC+ 
Test 1

In [100]:
here::i_am("rna/trajectories/infer_trajectory.R")

#suppressPackageStartupMessages(library(scran))
#suppressPackageStartupMessages(library(scater))
#suppressPackageStartupMessages(library(destiny))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code



In [101]:
## START TEST ##
args <- list()
args$sce <-file.path(io$basedir,"data/processed/rna/SingleCellExperiment.rds")
# args$metadata <- file.path(io$basedir,"results/atac/archR/celltype_assignment/sample_metadata_after_celltype_assignment.txt.gz")
args$metadata <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$trajectory_name <- "epiblast_blood"
args$celltype_label <- "celltype"
args$outdir <- file.path(io$basedir,"results/rna_atac/gene_regulatory_networks/scenicplus/blood")
args$n_pcs <- 25
args$sample <- c('E7.5_rep1','E7.5_rep2','E7.75_rep1','E8.0_rep1','E8.0_rep2','E8.5_rep1','E8.5_rep2','E8.75_rep1','E8.75_rep2')
## END TEST ##

# I/O
dir.create(args$outdir, showWarnings=F, recursive=T)

In [102]:
# Options
opts$celltype_trajectory_dic <- list(
  "epiblast_blood" =  c("Epiblast",
                        "Primitive_Streak" ,
                        "Nascent_mesoderm",
                        "ExE_mesoderm",
                        "Mixed_mesoderm",
                        "Allantois",
                        "Mesenchyme",
                        "Haematoendothelial_progenitors",
                        "Endothelium",
                        "Blood_progenitors_1",
                        "Blood_progenitors_2",
                        "Erythroid1",
                        "Erythroid2",
                        "Erythroid3"),
  "blood" = c("Haematoendothelial_progenitors", "Blood_progenitors_1", "Blood_progenitors_2", "Erythroid1", "Erythroid2", "Erythroid3"),
  "ectoderm" = c("Epiblast", "Rostral_neurectoderm", "Forebrain_Midbrain_Hindbrain"),
  "endoderm" = c("Epiblast", "Anterior_Primitive_Streak", "Def._endoderm", "Gut"),
  "mesoderm" = c("Epiblast", "Primitive_Streak", "Nascent_mesoderm"),
  "nmp" = c("NMP","Caudal_Mesoderm", "Somitic_mesoderm", "Spinal_cord")
  )

stopifnot(args$trajectory_name%in%names(opts$celltype_trajectory_dic))
opts$celltypes <- opts$celltype_trajectory_dic[[args$trajectory_name]]

opts$genes2plot <- list(
  "epiblast_blood" = c("Dnmt3b", "Mixl1", "Hbb-y","Etv2"),
  "blood" = c("Hbb-y","Etv2"),
  "ectoderm" = c("Utf1","Crabp1"),
  "endoderm" = c("Pou5f1","Krt8"),
  "mesoderm" = c("Dnmt3b","Mesp1"),
  "nmp" = c("Sox2","T")
)

stopifnot(args$trajectory_name%in%names(opts$genes2plot))
opts$genes_to_plot <- opts$genes2plot[[args$trajectory_name]]

In [105]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE]

stopifnot(args$celltype_label%in%colnames(sample_metadata))
sample_metadata <- sample_metadata   %>%
  .[,celltype:=eval(as.name(args$celltype_label))] %>%
  .[celltype%in%opts$celltypes] %>%
  .[,celltype:=factor(celltype,levels=opts$celltypes)] %>%
  .[sample%in% args$sample]

table(sample_metadata$celltype)

# Save
fwrite(sample_metadata, file.path(args$outdir,sprintf("%s_sample_metadata.txt.gz",args$trajectory_name)))


                      Epiblast               Primitive_Streak 
                          1101                            674 
              Nascent_mesoderm                   ExE_mesoderm 
                          1495                           1308 
                Mixed_mesoderm                      Allantois 
                           295                            741 
                    Mesenchyme Haematoendothelial_progenitors 
                          3539                           1019 
                   Endothelium            Blood_progenitors_1 
                           749                            272 
           Blood_progenitors_2                     Erythroid1 
                           673                           1243 
                    Erythroid2                     Erythroid3 
                          1029                            202 

## Create ATAC Matrix

In [119]:
atac_sce = readRDS(file.path(io$basedir, 'data/processed/atac/archR/Matrices/PeakMatrix_summarized_experiment.rds'))

In [121]:
cells_keep = colnames(atac_sce)[colnames(atac_sce) %in% sample_metadata$cell]
fwrite(sample_metadata[cell %in% cells_keep], file.path(args$outdir,sprintf("%s_sample_metadata.txt.gz",args$trajectory_name)))
fwrite(sample_metadata[cell %in% cells_keep], file.path(args$outdir,sprintf("%s_sample_metadata.tsv",args$trajectory_name)), sep='\t')

In [123]:
atac_sce = atac_sce[, cells_keep]

In [136]:
regions = rownames(atac_sce)
cells = colnames(atac_sce)
atac.mtx = atac_sce@assays@data$PeakMatrix
colnames(atac.mtx) = cells
rownames(atac.mtx) = regions

In [165]:
region_stats = data.table(region = rownames(atac.mtx), sum=rowSums(atac.mtx), var=rowVars(atac.mtx))

In [ ]:
ggarrange(gghistogram(region_stats$var), gghistogram(region_stats$sum))

In [166]:
region_keep = region_stats[sum>100 & var>0.1, region]

In [171]:
atac.dt = as.data.table(as.matrix(atac.mtx[region_keep,]), keep.rownames=T) %>% setnames('rn', 'cell')

In [172]:
fwrite(atac.dt, file.path(args$outdir,sprintf("%s_atac_mtx.tsv",args$trajectory_name)), sep='\t')

## RNA subset adata object in python?